# Course 1 — The Numerical Engine (NumPy)

Live-coding notebook that mirrors the slide deck. Run each cell as you teach.

**Sections**
1. The ND-Array (0:00–0:30)
2. Shape & Slicing (0:30–1:00)
3. Vectorization & Broadcasting (1:00–1:30)

In [ ]:
import numpy as np
rng = np.random.default_rng(0)   # seeded RNG — every run reproducible
np.__version__


## 1. The ND-Array

### List vs. ndarray — the memory story

In [ ]:
py_list = [1, 2, 3, 4]
arr = np.array([1, 2, 3, 4])
print(py_list, type(py_list))
print(arr, type(arr), arr.dtype)


### Why it's fast — measure it

In [ ]:
# Sum of one million ints
%timeit sum(range(1_000_000))
%timeit np.arange(1_000_000).sum()


### dtypes — one float promotes the whole array

In [ ]:
print(np.array([1, 2, 3]).dtype)
print(np.array([1.0, 2, 3]).dtype)
print(np.array([1, 2, 3], dtype=np.float32).dtype)


### Creating arrays

In [ ]:
print(np.zeros((2, 3)))
print(np.ones((2, 3), dtype=int))
print(np.arange(0, 10, 2))
print(np.linspace(0, 1, 5))
print(rng.normal(size=(2, 3)))


## 2. Shape & Slicing

In [ ]:
a = np.arange(12).reshape(3, 4)
print(a)
print('shape:', a.shape, 'ndim:', a.ndim, 'size:', a.size, 'dtype:', a.dtype)


### reshape with -1 — let NumPy compute the missing dimension

In [ ]:
x = np.arange(24)
print(x.reshape(2, 3, 4).shape)
print(x.reshape(2, -1).shape)


### Transpose & axis-swap (think batch-of-images conversions)

In [ ]:
img = rng.random((224, 224, 3))   # HWC layout (e.g. matplotlib)
print('HWC:', img.shape)
print('CHW:', img.transpose(2, 0, 1).shape)   # PyTorch layout


### Slicing in N dimensions

In [ ]:
a = np.arange(20).reshape(4, 5)
print(a)
print('row 1     :', a[1])
print('col 0     :', a[:, 0])
print('2x2 block :\n', a[1:3, 2:4])
print('reversed  :\n', a[::-1])


**Gotcha:** slices are *views*, not copies.

In [ ]:
b = a[:2, :2]
b[0, 0] = -999
print(a)   # the source mutated too!


### Boolean & fancy indexing

In [ ]:
x = rng.normal(size=10)
print('x         :', x)
print('positives :', x[x > 0])
x[x < 0] = 0
print('clipped   :', x)

ix = np.array([0, 3, 5])
print('picked    :', x[ix])


## 3. Vectorization & Broadcasting

### The loop you should never write

In [ ]:
x = rng.normal(size=1_000_000)

def slow(x):
    out = np.empty_like(x)
    for i in range(len(x)):
        out[i] = x[i]**2 + 3*x[i]
    return out

def fast(x):
    return x**2 + 3*x

%timeit slow(x)
%timeit fast(x)


### Reductions — learn the `axis` argument

In [ ]:
a = np.arange(12).reshape(3, 4)
print(a)
print('sum all   :', a.sum())
print('sum axis0 :', a.sum(axis=0))   # collapse rows -> per-column total
print('sum axis1 :', a.sum(axis=1))   # collapse cols -> per-row total


### Broadcasting — align shapes from the right

In [ ]:
row = np.array([1, 2, 3])
M = np.zeros((3, 3))
print(M + row)        # (3,) broadcast over (3, 3)

col = np.array([[10], [20], [30]])   # (3, 1)
print(col + row)      # (3, 1) + (3,) -> (3, 3)


### Worked example — pairwise distances without a single Python loop

In [ ]:
pts = rng.normal(size=(100, 2))
diff = pts[:, None, :] - pts[None, :, :]    # (100, 100, 2)
dist = np.sqrt((diff**2).sum(axis=-1))
print('dist matrix shape:', dist.shape)
print('symmetric? ', np.allclose(dist, dist.T))
print('zero diagonal?', np.allclose(np.diag(dist), 0))


### Recap
* ndarray is a contiguous typed buffer — that is the entire speed story.
* Reshape and transpose change *strides*, not data.
* Replace numeric `for` loops with broadcasting + reductions.

---

# Exercises

Each exercise below is followed by its solution.
Try to solve the tasks yourself before revealing the next cell.

---

## Exercise 1

# Exercise 1 — Array Basics

Goal: get fluent with creation, dtypes, and the shape vocabulary.

In [ ]:
import numpy as np
rng = np.random.default_rng(42)


**Task 1.** Create a `(4, 5)` array of zeros with dtype `int32`. Print its shape, dtype, and total memory in bytes.

In [ ]:
# your code here


**Task 2.** Build a 1-D array of 50 evenly-spaced values between -3 and 3 inclusive.

In [ ]:
# your code here


**Task 3.** Draw 10000 standard-normal samples. Compute the empirical mean and std and check they are close to 0 and 1.

In [ ]:
# your code here


**Task 4.** Make a `(3, 3)` identity matrix without using `np.eye`. (Hint: `np.arange` + boolean comparison.)

In [ ]:
# your code here


### Exercise 1 — Solution

# Exercise 1 — Solutions

In [ ]:
import numpy as np
rng = np.random.default_rng(42)


**Task 1.**

In [ ]:
a = np.zeros((4, 5), dtype=np.int32)
print(a.shape, a.dtype, a.nbytes, 'bytes')


**Task 2.**

In [ ]:
xs = np.linspace(-3, 3, 50)
print(xs[:5], '...', xs[-5:])
print('count:', xs.size)


**Task 3.**

In [ ]:
samples = rng.normal(size=10_000)
print('mean:', samples.mean(), '  std:', samples.std())
assert abs(samples.mean()) < 0.05
assert abs(samples.std() - 1) < 0.05


**Task 4.**

In [ ]:
n = 3
ix = np.arange(n)
I = (ix[:, None] == ix[None, :]).astype(int)
print(I)
assert np.array_equal(I, np.eye(n, dtype=int))


---

## Exercise 2

# Exercise 2 — Slicing & Reshape

We will treat an image as an ndarray — exactly what every CV model sees.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

# Synthetic 'image': a 64x64 grayscale gradient with a bright square in the center.
img = np.tile(np.linspace(0, 1, 64), (64, 1))
img[24:40, 24:40] = 1.0
plt.imshow(img, cmap='gray'); plt.title('source'); plt.axis('off');


**Task 1.** Crop the central 32×32 region of `img` and display it.

In [ ]:
# your code here


**Task 2.** Flip the image left↔right and top↔bottom (two separate outputs).

In [ ]:
# your code here


**Task 3.** Convert the image to RGB by stacking it 3 times along a new last axis. Print the new shape.

In [ ]:
# your code here


**Task 4.** Set every pixel whose value is above 0.9 to 0 (in a copy, not the original).

In [ ]:
# your code here


### Exercise 2 — Solution

# Exercise 2 — Solutions

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)
img = np.tile(np.linspace(0, 1, 64), (64, 1))
img[24:40, 24:40] = 1.0


**Task 1.** Center crop.

In [ ]:
crop = img[16:48, 16:48]
print(crop.shape)
plt.imshow(crop, cmap='gray'); plt.axis('off');


**Task 2.** Flips.

In [ ]:
lr = img[:, ::-1]
ud = img[::-1, :]
fig, ax = plt.subplots(1, 2, figsize=(6, 3))
ax[0].imshow(lr, cmap='gray'); ax[0].set_title('LR'); ax[0].axis('off')
ax[1].imshow(ud, cmap='gray'); ax[1].set_title('UD'); ax[1].axis('off');


**Task 3.** Grayscale -> RGB.

In [ ]:
rgb = np.stack([img, img, img], axis=-1)
print(rgb.shape)        # (64, 64, 3)


**Task 4.** Threshold on a copy.

In [ ]:
out = img.copy()
out[out > 0.9] = 0
print('max before:', img.max(), 'max after:', out.max())


---

## Exercise 3

# Exercise 3 — Vectorization & Broadcasting

Replace every `for` loop with a vectorized expression.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
x = rng.normal(size=1_000_000)


**Task 1.** Compute the per-element sigmoid `1 / (1 + exp(-x))` for `x` — no loops.

In [ ]:
# your code here


**Task 2.** Standardize a `(200, 5)` data matrix column-wise (subtract column mean, divide by column std).

In [ ]:
data = rng.normal(loc=5, scale=2, size=(200, 5))
# your code here


**Task 3.** Given 50 2-D points in `pts`, compute the full 50×50 pairwise Euclidean distance matrix. No loops.

In [ ]:
pts = rng.normal(size=(50, 2))
# your code here


**Task 4 (stretch).** Time a Python-loop version of Task 3 vs your vectorized version with `%timeit`.

In [ ]:
# your code here


### Exercise 3 — Solution

# Exercise 3 — Solutions

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
x = rng.normal(size=1_000_000)


**Task 1.**

In [ ]:
sig = 1 / (1 + np.exp(-x))
print(sig[:5])


**Task 2.** Column-wise standardize.

In [ ]:
data = rng.normal(loc=5, scale=2, size=(200, 5))
z = (data - data.mean(axis=0)) / data.std(axis=0)
print('column means ~0:', z.mean(axis=0).round(4))
print('column stds  ~1:', z.std(axis=0).round(4))


**Task 3.** Pairwise distances.

In [ ]:
pts = rng.normal(size=(50, 2))
diff = pts[:, None, :] - pts[None, :, :]
dist = np.sqrt((diff**2).sum(axis=-1))
print(dist.shape)
assert np.allclose(dist, dist.T)
assert np.allclose(np.diag(dist), 0)


**Task 4.** Timing showdown.

In [ ]:
def naive(pts):
    n = len(pts)
    out = np.empty((n, n))
    for i in range(n):
        for j in range(n):
            out[i, j] = np.sqrt(((pts[i] - pts[j])**2).sum())
    return out

def fast(pts):
    diff = pts[:, None, :] - pts[None, :, :]
    return np.sqrt((diff**2).sum(axis=-1))

%timeit naive(pts)
%timeit fast(pts)
